# Cross-dataset architecture validation

This notebook is the external-generalization gate for the architecture comparison. Each model is trained on one real dataset and evaluated on the other, in both directions. The target dataset is never used for training or normalization.

Models: compact CNN, small InceptionTime-style CNN, and MiniROCKET plus ridge. The CNNs use CUDA when available; MiniROCKET uses its standard CPU implementation.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sktime.transformations.panel.rocket import MiniRocketMultivariate
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
INTERIM = PROJECT_ROOT / 'data' / 'interim'
MAG_PATH = PROCESSED / 'validated_acceleration_magnitude_windows_float32.npy'
METADATA_PATH = PROCESSED / 'validated_window_metadata.csv'
SPLITS_PATH = INTERIM / 'participant_splits.csv'
FOLDS = 0
EPOCHS = 4
BATCH_SIZE = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(4)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
metadata = pd.read_csv(METADATA_PATH)
metadata['label_binary'] = metadata['label'].map({'healthy': 0, 'stroke': 1}).astype(int)
splits = pd.read_csv(SPLITS_PATH)
magnitude_windows = np.load(MAG_PATH, mmap_mode='r')
print('Device:', DEVICE)
print('Windows:', magnitude_windows.shape)

Device: cuda
Windows: (18511, 500, 3)


In [2]:
def source_indices(dataset_id):
    role_map = splits[splits['fold'].eq(FOLDS)].set_index('participant_key')['role']
    roles = metadata['participant_key'].map(role_map)
    train = np.flatnonzero((metadata['dataset_id'].eq(dataset_id) & roles.eq('training')).to_numpy())
    validation = np.flatnonzero((metadata['dataset_id'].eq(dataset_id) & roles.eq('validation')).to_numpy())
    return train, validation

def normalization(indices):
    total = np.zeros(3, dtype='float64')
    total_sq = np.zeros(3, dtype='float64')
    count = 0
    for start in range(0, len(indices), 512):
        batch = np.asarray(magnitude_windows[indices[start:start + 512]], dtype='float32')
        total += batch.sum(axis=(0, 1))
        total_sq += np.square(batch).sum(axis=(0, 1))
        count += batch.shape[0] * batch.shape[1]
    mean = total / count
    std = np.sqrt(np.maximum(total_sq / count - mean ** 2, 1e-8))
    return mean.astype('float32'), std.astype('float32')

def participant_weights(indices):
    frame = metadata.iloc[indices]
    participant_counts = frame.groupby('participant_key').size()
    class_counts = frame.groupby('label_binary').size()
    weights = frame['participant_key'].map(1.0 / participant_counts).to_numpy()
    weights *= frame['label_binary'].map(len(indices) / (2.0 * class_counts)).to_numpy()
    return (weights / weights.mean()).astype('float32')

class GaitDataset(Dataset):
    def __init__(self, indices, mean, std, weights=None):
        self.indices = np.asarray(indices, dtype='int64')
        self.mean = mean.reshape(1, 3)
        self.std = std.reshape(1, 3)
        self.weights = np.ones(len(self.indices), dtype='float32') if weights is None else weights
    def __len__(self): return len(self.indices)
    def __getitem__(self, item):
        index = int(self.indices[item])
        signal = ((np.asarray(magnitude_windows[index], dtype='float32') - self.mean) / self.std).T.copy()
        return torch.from_numpy(signal), torch.tensor(float(metadata.iloc[index]['label_binary'])), torch.tensor(float(self.weights[item])), torch.tensor(index)

def metrics_for(indices, probabilities):
    frame = metadata.iloc[np.asarray(indices)].copy()
    frame['probability'] = probabilities
    participant = frame.groupby(['participant_key', 'dataset_id', 'label_binary'], as_index=False)['probability'].mean()
    y = participant['label_binary'].to_numpy()
    p = participant['probability'].to_numpy()
    pred = (p >= 0.5).astype(int)
    return {'participants': len(participant), 'balanced_accuracy': balanced_accuracy_score(y, pred), 'roc_auc': roc_auc_score(y, p), 'f1': f1_score(y, pred)}, participant

In [3]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=16):
        super().__init__()
        bottleneck = min(32, in_channels)
        self.bottleneck = nn.Conv1d(in_channels, bottleneck, 1, bias=False)
        self.branches = nn.ModuleList([
            nn.Conv1d(bottleneck, out_channels, 7, padding=3, bias=False),
            nn.Conv1d(bottleneck, out_channels, 15, padding=7, bias=False),
            nn.Conv1d(bottleneck, out_channels, 25, padding=12, bias=False),
        ])
        self.pool_branch = nn.Conv1d(in_channels, out_channels, 1, bias=False)
        self.bn = nn.BatchNorm1d(out_channels * 4)
        self.residual = nn.Conv1d(in_channels, out_channels * 4, 1, bias=False) if in_channels != out_channels * 4 else nn.Identity()
    def forward(self, x):
        z = self.bottleneck(x)
        branches = [branch(z) for branch in self.branches]
        branches.append(self.pool_branch(nn.functional.max_pool1d(x, 3, stride=1, padding=1)))
        return nn.functional.gelu(self.bn(torch.cat(branches, dim=1)) + self.residual(x))

class CompactCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv1d(3, 32, 9, padding=4), nn.BatchNorm1d(32), nn.GELU(), nn.MaxPool1d(2), nn.Conv1d(32, 64, 7, padding=3), nn.BatchNorm1d(64), nn.GELU(), nn.MaxPool1d(2), nn.Conv1d(64, 128, 5, padding=2), nn.BatchNorm1d(128), nn.GELU(), nn.AdaptiveAvgPool1d(1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(128, 1))
    def forward(self, x): return self.classifier(self.features(x)).squeeze(1)

class InceptionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(InceptionBlock(3), nn.MaxPool1d(2), InceptionBlock(64), nn.AdaptiveAvgPool1d(1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(64, 1))
    def forward(self, x): return self.classifier(self.features(x)).squeeze(1)

def predict_model(model, loader):
    model.eval(); rows = []
    with torch.no_grad():
        for signals, _, _, indices in loader:
            p = torch.sigmoid(model(signals.to(DEVICE))).detach().cpu().numpy()
            rows.extend(zip(indices.numpy(), p))
    return np.array([p for _, p in rows], dtype='float32')

def train_torch_direction(train_dataset, test_dataset, factory, model_name):
    train_indices, validation_indices = source_indices(train_dataset)
    test_indices = np.flatnonzero(metadata['dataset_id'].eq(test_dataset).to_numpy())
    mean, std = normalization(train_indices)
    weights = participant_weights(train_indices)
    train_loader = DataLoader(GaitDataset(train_indices, mean, std, weights), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    validation_loader = DataLoader(GaitDataset(validation_indices, mean, std), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(GaitDataset(test_indices, mean, std), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = factory().to(DEVICE); optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_auc = -np.inf; best_state = None; patience = 2
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for signals, labels, batch_weights, _ in train_loader:
            optimizer.zero_grad(); logits = model(signals.to(DEVICE))
            loss = (nn.functional.binary_cross_entropy_with_logits(logits, labels.to(DEVICE), reduction='none') * batch_weights.to(DEVICE)).mean()
            loss.backward(); optimizer.step()
        val_p = predict_model(model, validation_loader)
        val_metrics, _ = metrics_for(validation_indices, val_p)
        print(f'{model_name} {train_dataset}->{test_dataset} epoch={epoch} internal_auc={val_metrics["roc_auc"]:.3f}')
        if val_metrics['roc_auc'] > best_auc:
            best_auc = val_metrics['roc_auc']; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}; patience = 2
        else:
            patience -= 1
            if patience == 0: break
    model.load_state_dict(best_state)
    test_p = predict_model(model, test_loader)
    test_metrics, test_participant = metrics_for(test_indices, test_p)
    test_metrics.update({'model': model_name, 'train_dataset': train_dataset, 'test_dataset': test_dataset, 'internal_validation_roc_auc': best_auc})
    test_participant['model'] = model_name; test_participant['train_dataset'] = train_dataset; test_participant['test_dataset'] = test_dataset
    return test_metrics, test_participant

In [4]:
directions = [('voisard_2025', 'felius_2024'), ('felius_2024', 'voisard_2025')]
torch_results = []; torch_predictions = []
for train_dataset, test_dataset in directions:
    for factory, name in [(CompactCNN, 'compact_cnn_gpu'), (InceptionCNN, 'inception_cnn_gpu')]:
        metrics, participant = train_torch_direction(train_dataset, test_dataset, factory, name)
        torch_results.append(metrics); torch_predictions.append(participant)
torch_results = pd.DataFrame(torch_results)
torch_predictions = pd.concat(torch_predictions, ignore_index=True)
print(torch_results.round(3).to_string(index=False))

compact_cnn_gpu voisard_2025->felius_2024 epoch=1 internal_auc=1.000
compact_cnn_gpu voisard_2025->felius_2024 epoch=2 internal_auc=1.000


compact_cnn_gpu voisard_2025->felius_2024 epoch=3 internal_auc=1.000


inception_cnn_gpu voisard_2025->felius_2024 epoch=1 internal_auc=1.000
inception_cnn_gpu voisard_2025->felius_2024 epoch=2 internal_auc=1.000


inception_cnn_gpu voisard_2025->felius_2024 epoch=3 internal_auc=1.000


compact_cnn_gpu felius_2024->voisard_2025 epoch=1 internal_auc=0.824


...

compact_cnn_gpu felius_2024->voisard_2025 epoch=4 internal_auc=0.920


inception_cnn_gpu felius_2024->voisard_2025 epoch=1 internal_auc=0.851


inception_cnn_gpu felius_2024->voisard_2025 epoch=2 internal_auc=0.858


inception_cnn_gpu felius_2024->voisard_2025 epoch=3 internal_auc=0.874


inception_cnn_gpu felius_2024->voisard_2025 epoch=4 internal_auc=0.885
 participants  balanced_accuracy  roc_auc    f1             model train_dataset test_dataset  internal_validation_roc_auc
          163              0.740    0.878 0.753   compact_cnn_gpu  voisard_2025  felius_2024                        1.000
          163              0.711    0.876 0.642 inception_cnn_gpu  voisard_2025  felius_2024                        1.000
          121              0.502    0.533 0.254   compact_cnn_gpu   felius_2024 voisard_2025                        0.920
          121              0.737    0.790 0.711 inception_cnn_gpu   felius_2024 voisard_2025                        0.885


In [5]:
rocket_results = []; rocket_predictions = []
for train_dataset, test_dataset in directions:
    train_indices, validation_indices = source_indices(train_dataset)
    test_indices = np.flatnonzero(metadata['dataset_id'].eq(test_dataset).to_numpy())
    mean, std = normalization(train_indices)
    train_x = ((np.asarray(magnitude_windows[train_indices], dtype='float32') - mean.reshape(1, 1, 3)) / std.reshape(1, 1, 3)).transpose(0, 2, 1)
    validation_x = ((np.asarray(magnitude_windows[validation_indices], dtype='float32') - mean.reshape(1, 1, 3)) / std.reshape(1, 1, 3)).transpose(0, 2, 1)
    test_x = ((np.asarray(magnitude_windows[test_indices], dtype='float32') - mean.reshape(1, 1, 3)) / std.reshape(1, 1, 3)).transpose(0, 2, 1)
    rocket = MiniRocketMultivariate(num_kernels=2000, max_dilations_per_kernel=16, n_jobs=1, random_state=42)
    train_features = rocket.fit_transform(train_x); validation_features = rocket.transform(validation_x); test_features = rocket.transform(test_x)
    classifier = RidgeClassifier(alpha=1.0)
    classifier.fit(train_features, metadata.iloc[train_indices]['label_binary'].to_numpy(), sample_weight=participant_weights(train_indices))
    val_decision = classifier.decision_function(validation_features); test_decision = classifier.decision_function(test_features)
    val_p = 1.0 / (1.0 + np.exp(-np.clip(val_decision, -30, 30))); test_p = 1.0 / (1.0 + np.exp(-np.clip(test_decision, -30, 30)))
    val_metrics, _ = metrics_for(validation_indices, val_p); test_metrics, test_participant = metrics_for(test_indices, test_p)
    test_metrics.update({'model': 'minirocket_ridge', 'train_dataset': train_dataset, 'test_dataset': test_dataset, 'internal_validation_roc_auc': val_metrics['roc_auc']})
    test_participant['model'] = 'minirocket_ridge'; test_participant['train_dataset'] = train_dataset; test_participant['test_dataset'] = test_dataset
    rocket_results.append(test_metrics); rocket_predictions.append(test_participant)
    print(f'minirocket {train_dataset}->{test_dataset} internal_auc={val_metrics["roc_auc"]:.3f} external_auc={test_metrics["roc_auc"]:.3f}')
rocket_results = pd.DataFrame(rocket_results)
rocket_predictions = pd.concat(rocket_predictions, ignore_index=True)

minirocket voisard_2025->felius_2024 internal_auc=0.976 external_auc=0.673


minirocket felius_2024->voisard_2025 internal_auc=0.939 external_auc=0.579


In [6]:
results = pd.concat([torch_results, rocket_results], ignore_index=True)
predictions = pd.concat([torch_predictions, rocket_predictions], ignore_index=True)
print(results.round(3).to_string(index=False))
results.to_csv(PROCESSED / 'cross_dataset_architecture_results.csv', index=False)
predictions.to_csv(PROCESSED / 'cross_dataset_architecture_predictions.csv', index=False)

 participants  balanced_accuracy  roc_auc    f1             model train_dataset test_dataset  internal_validation_roc_auc
          163              0.740    0.878 0.753   compact_cnn_gpu  voisard_2025  felius_2024                        1.000
          163              0.711    0.876 0.642 inception_cnn_gpu  voisard_2025  felius_2024                        1.000
          121              0.502    0.533 0.254   compact_cnn_gpu   felius_2024 voisard_2025                        0.920
          121              0.737    0.790 0.711 inception_cnn_gpu   felius_2024 voisard_2025                        0.885
          163              0.500    0.673 0.000  minirocket_ridge  voisard_2025  felius_2024                        0.976
          121              0.556    0.579 0.605  minirocket_ridge   felius_2024 voisard_2025                        0.939


## Decision rule

The preferred architecture must retain useful participant-level AUROC in both external directions. If every architecture is strongly asymmetric, the result is a data/protocol generalization problem rather than an architecture-selection problem.